<a id="top"></a>

# L13 lecture: What an eval run costs, and which scorer is worth it

```yaml
title: "L13 lecture: What an eval run costs, and which scorer is worth it"
keywords: evaluation, cost, tokens, sample size, scorer spectrum, llm-as-judge, human review, teacher lecture
estimated duration: 25
```

> **Lesson:** L13 Evaluation. Roadmap: [objectives.md](../../../../docs/origin/lesson_roadmaps/L13/objectives.md) (objective 4).
>
> Runs offline — the cost numbers are read off a deterministic `FakeModel` trace, and the LLM-judge uses an offline stub. An optional, gated cell runs the *real* judge if a key is present.

## Contents

- [1. Setup](#1-setup)
- [2. What an eval run costs — both ways](#2-what-an-eval-run-costs--both-ways)
  - [2.1 Back-of-envelope](#21-back-of-envelope)
  - [2.2 Ground it in real tokens](#22-ground-it-in-real-tokens)
  - [2.3 The sample-size trade-off](#23-the-sample-size-trade-off)
- [3. The scorer cost/judgment spectrum](#3-the-scorer-costjudgment-spectrum)
  - [3.1 A minimal LLM-as-judge](#31-a-minimal-llm-as-judge)
  - [3.2 (Optional, live) the real judge](#32-optional-live-the-real-judge)
- [4. Takeaways](#4-takeaways)

## 1. Setup

An eval run is **not free**: each case is a *full* `agent_loop.run(...)` — several model calls, not one — and you run N cases × K samples. To put a number on it we need a run to measure. We build one representative chaining run with the scripted `FakeModel` (which reports believable token `usage` on each `llm` span) and set **illustrative** prices.

> The prices below are *example* numbers for the arithmetic — check current model pricing before quoting a real figure.

In [1]:
from fluffy_potato_curriculum.common.agent_loop import RunResult, run
from fluffy_potato_curriculum.common.evals import EvalCase, EvalResult, evaluate
from fluffy_potato_curriculum.common.fake_model import (
    FakeModel,
    text_reply,
    tool_call,
    tool_reply,
)
from fluffy_potato_curriculum.common.tools import TOOLS

# Illustrative USD prices per 1M tokens (NOT live pricing — for the arithmetic only).
PRICE_IN = 3.0 / 1_000_000
PRICE_OUT = 15.0 / 1_000_000


def chaining_run_case(case: EvalCase) -> RunResult:
    """A clean calculator -> lookup -> answer run (3 model calls)."""
    model = FakeModel(
        [
            tool_reply(tool_call("c1", "calculator", {"expression": "17*23"})),
            tool_reply(tool_call("c2", "lookup", {"city": "Tokyo"})),
            text_reply("17 * 23 is 391, and Tokyo has a population of 37000000."),
        ]
    )
    return run(model, TOOLS, case.inputs["task"])


sample_case = EvalCase(id="chain", inputs={"task": "What is 17 * 23, and the population of Tokyo?"})
sample_run = chaining_run_case(sample_case)
model_calls = sum(1 for span in sample_run.trace if span.run_type == "llm")
print("model calls in one run:", model_calls)

model calls in one run: 3


[↑ Back to top](#top)

## 2. What an eval run costs — both ways

Quantify it **two ways**: a back-of-envelope formula, then the *actual* token numbers read off a trace. The estimate should be close enough to trust the formula; the real numbers keep you honest.

### 2.1 Back-of-envelope

The shape of the cost:

> `cost ≈ cases × samples × avg model-calls-per-run × per-call cost`

The key move most people miss: **each case is a full run** — several model calls, not one. Plug in a toy set (8 cases, 3 samples, ~3 calls/run) and a rough per-call cost.

In [2]:
n_cases = 8
samples = 3
calls_per_run = model_calls  # measured above: 3
rough_cost_per_call = 0.001  # a guess, in USD -- we replace this with a real number next

envelope = n_cases * samples * calls_per_run * rough_cost_per_call
total_calls = n_cases * samples * calls_per_run
print(f"{n_cases} cases x {samples} samples x {calls_per_run} calls = {total_calls} model calls")
print(f"back-of-envelope cost: ${envelope:.3f}")

8 cases x 3 samples x 3 calls = 72 model calls
back-of-envelope cost: $0.072


### 2.2 Ground it in real tokens

Now read the *actual* per-call token usage off the trace — the same `usage` fields L12 told you to trace and L01 taught you to cost — and compute a real per-call cost.

In [3]:
def run_cost(result: RunResult) -> float:
    """USD cost of one run, summed over the token usage on its llm spans."""
    in_tokens = sum(s.usage.input_tokens for s in result.trace if s.usage is not None)
    out_tokens = sum(s.usage.output_tokens for s in result.trace if s.usage is not None)
    return in_tokens * PRICE_IN + out_tokens * PRICE_OUT


cost_one_run = run_cost(sample_run)
real_cost_per_call = cost_one_run / model_calls
print(f"one run: ${cost_one_run:.6f}  ({model_calls} calls)")
print(f"real per-call cost: ${real_cost_per_call:.6f}")

# Re-estimate the full eval with the real per-call number, then GROUND it by
# summing the real cost of every sample run the harness actually executed.
grounded_estimate = n_cases * samples * calls_per_run * real_cost_per_call
print(f"\nre-estimated eval cost: ${grounded_estimate:.6f}")

cases = [EvalCase(id="chain", inputs={"task": "q"}) for _ in range(n_cases)]
report = evaluate(chaining_run_case, cases, [], samples=samples)
measured = sum(run_cost(s.run) for s in report.sample_results)
print(
    f"measured eval cost:    ${measured:.6f}  (summed over {len(report.sample_results)} real runs)"
)

one run: $0.001800  (3 calls)
real per-call cost: $0.000600

re-estimated eval cost: $0.043200
measured eval cost:    $0.043200  (summed over 24 real runs)


The estimate and the measured total line up — close enough to trust the formula, real enough not to hand-wave. On a *real* set (hundreds of cases, K=10, run on every commit) this number stops being cents.

### 2.3 The sample-size trade-off

More samples per case give a more trustworthy pass *rate* — but cost grows **linearly** in K. Choosing K is a deliberate cost/confidence trade, not "max it out."

In [4]:
for k in [1, 3, 5, 10]:
    cost_k = n_cases * k * calls_per_run * real_cost_per_call
    print(f"K={k:2}: {n_cases * k * calls_per_run:3} model calls, ${cost_k:.6f}")

K= 1:  24 model calls, $0.014400
K= 3:  72 model calls, $0.043200
K= 5: 120 model calls, $0.072000
K=10: 240 model calls, $0.144000


For a toy teaching set, K=3–5 is enough to make a rate legible. For a real set you'd weigh how flaky the cases are against the budget — a stable case barely needs sampling; a flaky one near a decision boundary may need K=10+.

[↑ Back to top](#top)

## 3. The scorer cost/judgment spectrum

Every scorer trades **cost** for **judgment**. There is no free, perfect, automatic check.

- table: scorers ordered by cost and by how much judgment each can exercise.

| Scorer | Cost | Judgment | Catches | Fails to catch |
| --- | --- | --- | --- | --- |
| **exact assertion** | ~free | none | exact matches | any rewording |
| **fuzzy / automated check** (substring, normalized, trajectory) | ~free | a little | the bug you encoded | qualities you didn't encode |
| **LLM-as-judge** | a model call | some (gameable, biased) | fuzzy qualities (tone, "graceful") | its own blind spots |
| **human review** | expensive, slow | the most | anything | scale |

Choosing a scorer is choosing a point on that curve for a given quality — a design decision, not a default.

### 3.1 A minimal LLM-as-judge

Some qualities no string check can express — *"did the answer give up **gracefully**?"* Consider two runs that both fail to get the answer: one apologizes clearly, one dumps an error. A substring check for the answer fails *both* identically; it can't tell them apart. An **LLM-as-judge** can.

Below, an offline **stub judge** stands in for the model call so the page runs deterministically (a real judge *is* a model call — see the next cell). The judge scorer is just another `Scorer`: it reads `run.final_text` and returns an `EvalResult`.

In [5]:
def graceful_run_case(case: EvalCase) -> RunResult:
    model = FakeModel(
        [
            tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://crash"})),
            text_reply(
                "I tried to fetch that page but it kept failing, so I could not get the answer -- sorry about the trouble."
            ),
        ]
    )
    return run(model, TOOLS, case.inputs["task"])


def blunt_run_case(case: EvalCase) -> RunResult:
    model = FakeModel(
        [
            tool_reply(tool_call("c1", "flaky_fetch", {"url": "https://crash"})),
            text_reply("RuntimeError: connection reset by peer"),
        ]
    )
    return run(model, TOOLS, case.inputs["task"])


JUDGE_PROMPT = (
    "You are grading an AI agent's final reply on ONE quality: did it give up "
    "GRACEFULLY -- acknowledge the failure clearly and politely -- rather than "
    "dumping a raw error? Reply with exactly PASS or FAIL.\n\nReply:\n"
)


def stub_judge(final_text: str) -> str:
    """An OFFLINE stand-in for a model call: PASS if the reply reads as a graceful
    apology, else FAIL. A real judge would send JUDGE_PROMPT + final_text to a model."""
    graceful = any(word in final_text.lower() for word in ("sorry", "apolog", "could not", "tried"))
    return "PASS" if graceful else "FAIL"


def gave_up_gracefully(*, run: RunResult, example: EvalCase) -> EvalResult:
    """LLM-as-judge scorer (offline stub): score a fuzzy quality no substring catches."""
    verdict = stub_judge(run.final_text)
    return EvalResult(key="graceful", score=verdict == "PASS", comment=verdict)


case = EvalCase(id="giveup", inputs={"task": "Fetch https://crash and tell me the answer."})
for label, run_case in [("graceful", graceful_run_case), ("blunt", blunt_run_case)]:
    report = evaluate(run_case, [case], [gave_up_gracefully])
    verdict = report.sample_results[0].results[0]
    print(f"{label:9} -> graceful={verdict.score}  ({verdict.comment})")

graceful  -> graceful=True  (PASS)
blunt     -> graceful=False  (FAIL)


The judge scores `graceful=True` for the apology and `graceful=False` for the error dump — a quality no substring or normalized check could express. But the judge has **its own costs**: it's a model call (so it costs tokens and latency), and it can be wrong, biased, or gamed. The L13 judge is a one-screen illustration; a **later at-scale eval lesson** unpacks what an LLM-judge can and can't reliably score.

### 3.2 (Optional, live) the real judge

The cell below runs the *actual* judge through the `potato_llm` Anthropic client (a plain chat call — no tools). It is **gated on `ANTHROPIC_API_KEY`** via the config seam, so a keyless run prints a note and skips.

In [6]:
from fluffy_potato_curriculum.common.config import get_settings

if get_settings().anthropic_api_key:
    from fluffy_potato_curriculum.potato_llm import AnthropicClient, Message

    judge_client = AnthropicClient(model="claude-sonnet-4-6")

    def live_judge(final_text: str) -> str:
        reply = judge_client.chat(
            [Message.system(JUDGE_PROMPT), Message.user(final_text)],
            max_tokens=8,
        )
        return "PASS" if "PASS" in reply.text.upper() else "FAIL"

    for label, run_case in [("graceful", graceful_run_case), ("blunt", blunt_run_case)]:
        final_text = run_case(case).final_text
        print(f"{label:9} -> live judge says {live_judge(final_text)}")
else:
    print("No ANTHROPIC_API_KEY set — skipping the live judge (the stub above made the point).")

No ANTHROPIC_API_KEY set — skipping the live judge (the stub above made the point).


[↑ Back to top](#top)

## 4. Takeaways

- **An eval run is not free.** It's N cases × K samples × *several* model calls each. Estimate it with `cases × samples × calls/run × per-call cost`, then ground the estimate in the real token `usage` on a trace.
- **Sample size is a deliberate trade.** Cost grows linearly in K; more samples buy a more trustworthy rate. Pick K for how flaky the case is and what the budget allows — not "max it out."
- **Every scorer trades cost for judgment.** Exact assertions are cheap and dumb; humans are expensive and wise; the LLM-as-judge sits between, with its *own* error modes. Choosing a scorer is choosing a point on that curve.
- **The LLM-judge earns its place only on qualities cheaper checks can't express** (tone, "graceful"). It's a fallible scorer, not an oracle — a later at-scale eval lesson treats it properly.
- **Next:** [`L1307_lecture`](L1307_lecture.md) — the short closer: carry the eval set forward into every agent you build (L04, L11, and on to evaluation-at-scale later).

[↑ Back to top](#top)